# Healthcare KG Population Intelligence — V4.1
**Independent survey audit | missingness-adjustment uncertainty | comparability gate**

This is a **standalone, isolated Colab notebook** derived from the supplied V4 source, with the 2017–2018 V3.1 reference retained. It retrieves public NHANES data afresh, uses the supplied baseline KG only for a read-only checksum, and writes results solely under `/content/Healthcare_KG_Population_Intelligence_V4_1`. **No GitHub calls.**

**Interpretation guardrails:** one-visit CKD screen, approximate type-2 diabetes classification, not verified CKD chronicity, not Kerendia eligibility. 2021–2023 is a nonstandard 2-year-interval release (Aug 2021–Aug 2023); do not infer trends without CDC documentation and laboratory harmonization review. The V4 complete-case estimand and response-adjusted estimand are different; both are reported.

In [ ]:
%pip -q install pandas numpy scipy requests pyreadstat matplotlib

## 1. Load baseline and initialize

In [ ]:
from pathlib import Path
import os, json, hashlib, zipfile, datetime, io, time
import numpy as np, pandas as pd, requests
from scipy.stats import t as student_t
ROOT=Path('/content/Healthcare_KG_Population_Intelligence_V4_1');ROOT.mkdir(parents=True,exist_ok=True)
UPLOAD_INTERACTIVE=True
INCLUDE_2021_2023=True  # change to False if CDC variable harmonization fails
if UPLOAD_INTERACTIVE:
    from google.colab import files
    uploaded=files.upload()
    incoming=Path(next(iter(uploaded)))
else:
    incoming=Path('/content/v3_1_outputs.zip')
if incoming.suffix.lower()=='.zip':
    with zipfile.ZipFile(incoming) as archive:
        names={Path(x).name:x for x in archive.namelist() if not x.endswith('/')}
        assert 'baseline_kg_COPY.json' in names, 'Missing baseline_kg_COPY.json'
        (ROOT/'baseline_kg_COPY.json').write_bytes(archive.read(names['baseline_kg_COPY.json']))
        prior_audit=next((x for x in names if 'audit' in x.lower() and x.endswith('.json')),None)
        if prior_audit:(ROOT/'v3_1_previous_audit.json').write_bytes(archive.read(names[prior_audit]))
else:
    assert incoming.name=='baseline_kg_COPY.json'
    (ROOT/'baseline_kg_COPY.json').write_bytes(incoming.read_bytes())
kg_path=ROOT/'baseline_kg_COPY.json'
KG_SHA256=hashlib.sha256(kg_path.read_bytes()).hexdigest()
print('Read-only baseline:',kg_path.name,'SHA256:',KG_SHA256)

## 2. Retrieve NHANES cycle data and log provenance

In [ ]:
CYCLES={
 '2015-2016':{'year':'2015','suffix':'I','period_years':2,'era':'pre-pandemic'},
 '2017-2018':{'year':'2017','suffix':'J','period_years':2,'era':'pre-pandemic'},
}
if INCLUDE_2021_2023:
 CYCLES['2021-2023']={'year':'2021','suffix':'L','period_years':2,'era':'post-pandemic-nonstandard'}
COMPONENTS=['DEMO','ALB_CR','BIOPRO','DIQ']
session=requests.Session(); session.headers['User-Agent']='HealthcareKG-V4-research/1.0'
manifest=[]; cycles={}; errors={}
for label,cfg in CYCLES.items():
  tables={}
  try:
    for component in COMPONENTS:
      name=f'{component}_{cfg["suffix"]}'
      url=f'https://wwwn.cdc.gov/Nchs/Data/Nhanes/Public/{cfg["year"]}/DataFiles/{name}.xpt'
      response=session.get(url,timeout=120);response.raise_for_status()
      if len(response.content)<1000:raise ValueError('Unexpectedly small XPT: '+name)
      local=ROOT/(name+'.xpt');local.write_bytes(response.content)
      df=pd.read_sas(local,format='xport')
      if 'SEQN' not in df:raise ValueError(f'{name}: SEQN missing; check CDC file/version')
      tables[component]=df
      manifest.append({'cycle':label,'component':name,'url':url,'sha256':hashlib.sha256(response.content).hexdigest(),'n_rows':len(df),'bytes':len(response.content),'status':'OK'})
    cycles[label]=tables
    print(label,[(k,len(v)) for k,v in tables.items()])
  except Exception as exc:
    errors[label]=str(exc)
    print('ERROR:',label,str(exc),'— this cycle will NOT be included')
    if label=='2017-2018':raise
assert '2017-2018' in cycles
pd.DataFrame(manifest).to_csv(ROOT/'v4_source_manifest.csv',index=False)

## 3. Harmonize with V4 definitions (unchanged)

In [ ]:
REQUIRED_DEMO=['SEQN','RIDAGEYR','RIAGENDR','RIDRETH3','WTMEC2YR','SDMVSTRA','SDMVPSU']
REQUIRED_LABS=['URXUMA','URXUCR','LBXSCR']
REQUIRED_DIQ=['DIQ010','DID040']
def harmonize(label,tables):
  for name,fields in [('DEMO',REQUIRED_DEMO),('ALB_CR',['URXUMA','URXUCR']),('BIOPRO',['LBXSCR']),('DIQ',REQUIRED_DIQ)]:
    absent=[f for f in fields if f not in tables[name].columns]
    if absent:raise ValueError(f'{label} {name} missing {absent}; cannot assume equivalence')
  d=tables['DEMO'][REQUIRED_DEMO].copy()
  for component in ['ALB_CR','BIOPRO','DIQ']:
    d=d.merge(tables[component],on='SEQN',how='left',validate='one_to_one')
  d=d.loc[pd.to_numeric(d.RIDAGEYR,errors='coerce')>=20].copy()
  for col in REQUIRED_DEMO+REQUIRED_LABS+REQUIRED_DIQ:
    d[col]=pd.to_numeric(d[col],errors='coerce')
  female=d.RIAGENDR.eq(2)
  k=np.where(female,.7,.9);a=np.where(female,-.241,-.302)
  r=d.LBXSCR/k
  d['egfr']=142*np.minimum(r,1)**a*np.maximum(r,1)**(-1.2)*(.9938**d.RIDAGEYR)*np.where(female,1.012,1)
  d.loc[d.LBXSCR.isna()|d.LBXSCR.le(0),'egfr']=np.nan
  d['uacr']=100*d.URXUMA/d.URXUCR
  d.loc[d.URXUMA.isna()|d.URXUCR.isna()|d.URXUCR.le(0),'uacr']=np.nan
  d['ckd_screen']=np.where(d.egfr.notna()&d.uacr.notna(),(d.egfr<60)|(d.uacr>=30),np.nan)
  d['diabetes_reported']=np.where(d.DIQ010.isin([1,2]),d.DIQ010.eq(1).astype(float),np.nan)
  d['t2d_proxy']=np.where(d.diabetes_reported.eq(0),0,np.where(d.diabetes_reported.eq(1)&d.DID040.between(1,100),(d.DID040>=30).astype(float),np.nan))
  d['joint']=np.where(d.ckd_screen.notna()&pd.notna(d.t2d_proxy),(d.ckd_screen==1)&(d.t2d_proxy==1),np.nan)
  d['complete']=d.joint.notna()
  d['case']=d.joint.eq(1).astype(int)
  d['eligible_design']=d.WTMEC2YR.gt(0)&d.SDMVSTRA.notna()&d.SDMVPSU.notna()
  d['age_group']=pd.cut(d.RIDAGEYR,[20,45,65,75,np.inf],right=False,labels=['20-44','45-64','65-74','75+'])
  d['sex']=d.RIAGENDR.map({1:'Male',2:'Female'})
  d['race_ethnicity_code']=d.RIDRETH3.astype('Int64').astype(str)
  d['cycle']=label
  return d
cohorts={}
for label,tables in cycles.items():
 try:
  cohorts[label]=harmonize(label,tables)
  print(label,'adults',len(cohorts[label]),'complete',int((cohorts[label].complete&cohorts[label].eligible_design).sum()),'cases',int((cohorts[label].complete&cohorts[label].eligible_design&cohorts[label].case.eq(1)).sum()))
 except Exception as exc:
  errors[label]=str(exc);print('HARMONIZATION FAILED:',label,str(exc))
assert '2017-2018' in cohorts

## 4. Reproduce V4 survey estimates

In [ ]:
def survey_total_var(df,quantity):
    x=df[['SDMVSTRA','SDMVPSU']].copy()
    x['v']=df.WTMEC2YR.to_numpy()*np.asarray(quantity,dtype=float)
    sums=x.groupby(['SDMVSTRA','SDMVPSU'],observed=True).v.sum()
    variance=0.
    for _,g in sums.groupby(level=0):
        a=g.to_numpy();m=len(a)
        if m>1:variance+=m/(m-1)*np.square(a-a.mean()).sum()
    return float(variance)
def survey_stats(d,case_mask,denom_mask):
    z=d.loc[d.eligible_design].copy()
    c=np.asarray(case_mask.reindex(z.index).fillna(False),dtype=float)
    den=np.asarray(denom_mask.reindex(z.index).fillna(False),dtype=float)
    n=float((z.WTMEC2YR*c).sum());tot=float((z.WTMEC2YR*den).sum())
    strata=z.groupby('SDMVSTRA',observed=True).SDMVPSU.nunique()
    dfree=int(np.maximum(strata-1,0).sum())
    if not tot or not dfree:raise ValueError('Empty denominator or invalid survey design')
    p=n/tot;vc=survey_total_var(z,c);vr=survey_total_var(z,c-p*den)
    tcrit=student_t.ppf(.975,dfree)
    return {'weighted_cases':n,'weighted_denominator':tot,'prevalence':p,'count_variance':vc,'prevalence_variance':vr/tot**2,'count_ci_low':max(0,n-tcrit*np.sqrt(vc)),'count_ci_high':n+tcrit*np.sqrt(vc),'prevalence_ci_low':max(0,p-tcrit*np.sqrt(vr)/tot),'prevalence_ci_high':min(1,p+tcrit*np.sqrt(vr)/tot),'design_df':dfree,'unweighted_cases':int(c.sum()),'unweighted_denominator':int(den.sum())}
rows=[]
for label,d in cohorts.items():
  complete=d.complete&d.eligible_design
  case=complete&d.case.eq(1)
  s=survey_stats(d,case,complete)
  rows.append({'cycle':label,'method':'complete_case',**s})
  if label=='2017-2018':
    print('V3.1 comparability:',s['weighted_cases'],'(reference ≈ 9,497,620; deviations require investigation)')
main=pd.DataFrame(rows)
main.to_csv(ROOT/'v4_population_estimates.csv',index=False)
display(main[['cycle','weighted_cases','prevalence','count_ci_low','count_ci_high','unweighted_cases','unweighted_denominator']])

## 5. Reproduce V4 demographic response adjustment

In [ ]:
adjusted=[];missingness=[]
for label,d in cohorts.items():
 z=d.loc[d.eligible_design].copy()
 z['responded']=z.complete.astype(int)
 z['stratum_cell']=z.age_group.astype(str)+'|'+z.sex.astype(str)+'|'+z.race_ethnicity_code
 for group in ['age_group','sex','race_ethnicity_code']:
  for value,sub in z.groupby(group,observed=True,dropna=False):
   all_w=sub.WTMEC2YR.sum();observed_w=sub.loc[sub.complete,'WTMEC2YR'].sum()
   missingness.append({'cycle':label,'dimension':group,'group':str(value),'n':len(sub),'weighted_missing_fraction':float(1-observed_w/all_w) if all_w else np.nan})
 # Coarsen empty/small cells using age x sex, then overall (document fallback counts).
 groupcols=[['age_group','sex','race_ethnicity_code'],['age_group','sex'],[]]
 response=np.full(len(z),np.nan)
 fallback=np.full(len(z),-1)
 for level,cols in enumerate(groupcols):
  if cols:
   g=z.groupby(cols,observed=True,dropna=False)
   totals=g.WTMEC2YR.transform('sum')
   # Map group totals using per-row keys to avoid categorical indexing quirks.
   obs_w=g.WTMEC2YR.transform(lambda s: float((s*z.loc[s.index,'complete']).sum()))
   n_obs=g.complete.transform('sum')
  else:
   totals=pd.Series(z.WTMEC2YR.sum(),index=z.index)
   obs_w=pd.Series(z.loc[z.complete,'WTMEC2YR'].sum(),index=z.index)
   n_obs=pd.Series(z.complete.sum(),index=z.index)
  good=(n_obs>=10)&(obs_w>0)&(totals>0)
  fill=np.isnan(response)&good.to_numpy()
  response[fill]=(obs_w.to_numpy()[fill]/totals.to_numpy()[fill])
  fallback[fill]=level
 if np.isnan(response).any():raise ValueError('Unresolved response-propensity cells')
 z['response_probability']=response
 z['adjusted_weight']=np.where(z.complete,z.WTMEC2YR/response,0.)
 pop=float((z.adjusted_weight*z.case).sum())
 allpop=float(z.WTMEC2YR.sum())
 adjusted.append({'cycle':label,'method':'cell_response_adjustment','weighted_cases':pop,'weighted_denominator':allpop,'prevalence':pop/allpop,'min_response_probability':float(response.min()),'max_weight_multiplier':float((1/response).max()),'n_fallback_to_age_sex':int((fallback==1).sum()),'n_fallback_to_overall':int((fallback==2).sum()),'variance_status':'NOT ESTIMATED: propensity uncertainty not incorporated'})
 pd.DataFrame({'SEQN':z.SEQN,'cycle':label,'response_probability':response,'fallback_level':fallback}).to_csv(ROOT/f'v4_response_diagnostics_{label}.csv',index=False)
adjusted_df=pd.DataFrame(adjusted)
adjusted_df.to_csv(ROOT/'v4_missingness_adjusted_sensitivity.csv',index=False)
pd.DataFrame(missingness).to_csv(ROOT/'v4_missingness_by_demographics.csv',index=False)
display(adjusted_df[['cycle','weighted_cases','prevalence','min_response_probability','max_weight_multiplier']])

## 6. Independent R `survey` check on all completed cycles
This installs R and the CRAN `survey` package if needed. If R is unavailable, the notebook **flags the validation as incomplete** rather than claiming a pass.

In [ ]:
import shutil, subprocess
from pathlib import Path
if shutil.which('Rscript') is None:
    print('Installing R...')
    subprocess.run(['apt-get','update','-qq'],check=True)
    subprocess.run(['apt-get','install','-y','-qq','r-base-core','r-cran-survey'],check=True)
check=subprocess.run(['Rscript','-e','if(!requireNamespace("survey",quietly=TRUE)) quit(status=3)'],capture_output=True,text=True)
if check.returncode!=0:
    print('Installing R survey from apt...')
    subprocess.run(['apt-get','install','-y','-qq','r-cran-survey'],check=True)
assert subprocess.run(['Rscript','-e','stopifnot(requireNamespace("survey",quietly=TRUE))']).returncode==0
print('R survey available')

In [ ]:
# Export full adult MEC design domains; outcomes are zero outside complete cases.
for label,d in cohorts.items():
    z=d.loc[d.eligible_design,['SDMVSTRA','SDMVPSU','WTMEC2YR','complete','case']].copy()
    z['case_indicator']=(z.complete & z.case.eq(1)).astype(int)
    z['denominator_indicator']=z.complete.astype(int)
    z.to_csv(ROOT/f'v4_1_r_input_{label}.csv',index=False)
r_code=r'''suppressPackageStartupMessages(library(survey))
options(survey.lonely.psu="fail")
a=commandArgs(trailingOnly=TRUE)
d=read.csv(a[1]); d$case_indicator=as.numeric(d$case_indicator); d$denominator_indicator=as.numeric(d$denominator_indicator)
D=svydesign(ids=~SDMVPSU,strata=~SDMVSTRA,weights=~WTMEC2YR,nest=TRUE,data=d)
N=svytotal(~case_indicator,D); P=svyratio(~case_indicator,~denominator_indicator,D)
res=data.frame(weighted_cases=as.numeric(coef(N)[1]),count_variance=as.numeric(vcov(N)[1,1]),prevalence=as.numeric(coef(P)[1]),prevalence_variance=as.numeric(vcov(P)[1,1]),design_df=degf(D))
write.csv(res,a[2],row.names=FALSE)
'''
(ROOT/'v4_1_r_validation.R').write_text(r_code)
rchecks=[]
for label in cohorts:
    output=ROOT/f'v4_1_r_result_{label}.csv'
    cmd=['Rscript',str(ROOT/'v4_1_r_validation.R'),str(ROOT/f'v4_1_r_input_{label}.csv'),str(output)]
    run=subprocess.run(cmd,capture_output=True,text=True)
    if run.returncode:
        rchecks.append({'cycle':label,'status':'R_FAILED','error':run.stderr[-1200:]});continue
    r=pd.read_csv(output).iloc[0];p=main.set_index('cycle').loc[label]
    row={'cycle':label,'status':'PASS'}
    for col in ['weighted_cases','count_variance','prevalence','prevalence_variance']:
        row['python_'+col]=float(p[col]);row['r_'+col]=float(r[col]);row['relative_error_'+col]=abs(float(p[col])-float(r[col]))/max(abs(float(r[col])),1e-12)
        if row['relative_error_'+col]>1e-5:row['status']='REVIEW'
    row['r_design_df']=int(r.design_df);row['python_design_df']=int(p.design_df)
    if row['r_design_df']!=row['python_design_df']:row['status']='REVIEW'
    rchecks.append(row)
r_df=pd.DataFrame(rchecks);r_df.to_csv(ROOT/'v4_1_independent_r_validation.csv',index=False)
display(r_df)

## 7. Response-adjusted variance: design-based PSU jackknife with **re-fitting**

For each replicate, delete one PSU **within its stratum**, multiply weights of remaining PSUs in that stratum by `m/(m-1)`, **recompute demographic response propensities**, then recompute the adjusted count. This propagates uncertainty in the estimated response rates and the outcome. The jackknife variance is `sum_h (m_h-1)/m_h * sum_j (theta_hj - mean_j(theta_hj))²`.

This is a sensitivity variance estimate, **not a guarantee of unbiasedness under missing-not-at-random**. Response-cell instability, calibration and CDC weight guidance still require review.

In [ ]:
def response_adjusted_total(z,weights):
    # Same hierarchical response cells and minimum observed count as V4.
    response=np.full(len(z),np.nan,dtype=float)
    fallback=np.full(len(z),-1,dtype=int)
    for level,cols in enumerate([['age_group','sex','race_ethnicity_code'],['age_group','sex'],[]]):
        tmp=z[cols].copy() if cols else pd.DataFrame(index=z.index)
        tmp['w']=np.asarray(weights,float);tmp['obs_w']=tmp.w*z.complete.to_numpy().astype(float)
        tmp['obs_n']=(z.complete.to_numpy() & (np.asarray(weights,float)>0)).astype(int)
        if cols:
            g=tmp.groupby(cols,observed=True,dropna=False)
            total=g.w.transform('sum').to_numpy(); observed=g.obs_w.transform('sum').to_numpy(); count=g.obs_n.transform('sum').to_numpy()
        else:
            total=np.full(len(z),tmp.w.sum());observed=np.full(len(z),tmp.obs_w.sum());count=np.full(len(z),tmp.obs_n.sum())
        ok=(count>=10)&(observed>0)&(total>0)
        fill=np.isnan(response)&ok
        response[fill]=observed[fill]/total[fill];fallback[fill]=level
    if np.isnan(response).any():raise ValueError('Unresolved propensity cells in replicate')
    estimate=np.sum(np.asarray(weights,float)*z.complete.to_numpy()*z.case.to_numpy()/response)
    return float(estimate),int(np.sum(fallback==1)),int(np.sum(fallback==2)),float(response.min())

jk_rows=[];jk_replicates=[]
for label,d in cohorts.items():
    z=d.loc[d.eligible_design].copy().reset_index(drop=True)
    w=z.WTMEC2YR.to_numpy(float)
    full,_,_,min_resp=response_adjusted_total(z,w)
    replicates=[]
    for h,g in z.groupby('SDMVSTRA',observed=True):
        psus=sorted(g.SDMVPSU.dropna().unique());m=len(psus)
        if m<2:raise ValueError(f'{label}: lonely PSU in stratum {h}')
        for j in psus:
            wr=w.copy();same=z.SDMVSTRA.eq(h).to_numpy();deleted=same & z.SDMVPSU.eq(j).to_numpy()
            wr[same]*=m/(m-1);wr[deleted]=0
            est,fb1,fb2,mr=response_adjusted_total(z,wr)
            replicates.append((h,j,m,est))
            jk_replicates.append({'cycle':label,'stratum':h,'deleted_psu':j,'n_psu_in_stratum':m,'replicate_estimate':est,'fallback_age_sex':fb1,'fallback_overall':fb2,'min_response':mr})
    var=0.
    for h in set(x[0] for x in replicates):
        arr=np.array([x[3] for x in replicates if x[0]==h]);m=len(arr)
        var+=(m-1)/m*np.square(arr-arr.mean()).sum()
    dfree=sum(len(g.SDMVPSU.unique())-1 for _,g in z.groupby('SDMVSTRA'))
    crit=student_t.ppf(.975,dfree)
    jk_rows.append({'cycle':label,'adjusted_count':full,'jackknife_se':np.sqrt(var),'jackknife_variance':var,'count_ci_low':max(0,full-crit*np.sqrt(var)),'count_ci_high':full+crit*np.sqrt(var),'design_df':dfree,'replicates':len(replicates),'min_response_probability':min_resp,'status':'SENSITIVITY_NOT_CAUSAL'})
jk_df=pd.DataFrame(jk_rows);jk_df.to_csv(ROOT/'v4_1_adjusted_jackknife.csv',index=False)
pd.DataFrame(jk_replicates).to_csv(ROOT/'v4_1_jackknife_replicates.csv',index=False)
display(jk_df)

## 8. Laboratory/weight comparability checklist and epidemiological review

Automated checks establish **variable presence, units claimed in documentation, and arithmetic consistency**; they cannot prove clinical comparability or secular trends. **2021–2023 weighting is a separate explicit gate.** Check the CDC analytic guidelines and laboratory method documentation for every period. No pooled trend inference or market sizing is authorized here.

In [ ]:
# Audit required data and plausible ranges; do not silently correct or exclude anomalies.
comparability=[]
for label,d in cohorts.items():
    cfg=CYCLES[label]
    comparability.append({'cycle':label,'weight_variable':'WTMEC2YR','period_years_config':cfg['period_years'],
      'n_adults':len(d),'n_valid_mec_design':int(d.eligible_design.sum()),
      'n_complete':int((d.eligible_design&d.complete).sum()),
      'serum_creatinine_missing':int(d.LBXSCR.isna().sum()),'urine_albumin_missing':int(d.URXUMA.isna().sum()),
      'urine_creatinine_missing':int(d.URXUCR.isna().sum()),
      'serum_creatinine_median':float(d.LBXSCR.median()),'uacr_median':float(d.uacr.median()),
      'weight_guidance_verified':False,'lab_method_equivalence_verified':False,
      'race_ethnicity_harmonization_verified':False,
      'trend_ready':False,'review_note':'Manual CDC documentation verification required; 2021-2023 nonstandard release.'})
comp=pd.DataFrame(comparability);comp.to_csv(ROOT/'v4_1_cycle_comparability_audit.csv',index=False)
display(comp)

## 9. Baseline checksum, validation gates, and outputs

In [ ]:
reference=main.set_index('cycle').loc['2017-2018','weighted_cases']
kg_intact=hashlib.sha256(kg_path.read_bytes()).hexdigest()==KG_SHA256
r_pass=bool(len(r_df)==len(cohorts) and r_df.status.eq('PASS').all())
jk_pass=bool(len(jk_df)==len(cohorts) and np.isfinite(jk_df.jackknife_se).all())
report={'version':'4.1','created_utc':datetime.datetime.now(datetime.timezone.utc).isoformat(),
 'development_basis':'V4 source notebook, V3.1 Gemini-corrected methodology reference',
 'baseline_kg_sha256':KG_SHA256,'baseline_unchanged':kg_intact,'github_accessed':False,
 'cycles_completed':list(cohorts),'cycle_errors':errors,
 'v3_1_reference_count':9497620.0,'reproduced_2017_2018_count':float(reference),
 'v3_1_reference_within_0_1pct':bool(abs(reference-9497620)/9497620<.001),
 'independent_r_survey_pass':r_pass,'response_adjusted_jackknife_completed':jk_pass,
 'cdc_weight_guidance_verified':False,'cross_cycle_lab_comparability_verified':False,
 'clinical_ckd_chronicity_confirmed':False,'true_t2d_classification_confirmed':False,
 'statistical_gate':'PASS' if r_pass and jk_pass and kg_intact else 'REVIEW_REQUIRED',
 'epidemiological_gate':'REVIEW_REQUIRED',
 'overall_gate':'REVIEW_REQUIRED',
 'notes':['Do not treat response adjustment as MNAR correction.',
          'Do not interpret cross-cycle differences as trends without CDC comparability review.',
          'Not a Kerendia-eligible population estimate.']}
(ROOT/'v4_1_validation_report.json').write_text(json.dumps(report,indent=2))
print(json.dumps(report,indent=2))
assert kg_intact
with zipfile.ZipFile(ROOT/'Healthcare_KG_Population_Intelligence_V4_1_Outputs.zip','w',zipfile.ZIP_DEFLATED) as archive:
    for p in ROOT.glob('v4_1_*'):
        if p.suffix in ('.csv','.json','.R'):archive.write(p,arcname=p.name)
    for p in ROOT.glob('v4_population_estimates.csv'):archive.write(p,arcname=p.name)
    for p in ROOT.glob('v4_missingness_adjusted_sensitivity.csv'):archive.write(p,arcname=p.name)
    archive.write(kg_path,arcname=kg_path.name)
print('Ready:',ROOT/'Healthcare_KG_Population_Intelligence_V4_1_Outputs.zip')

## 10. Download results

In [ ]:
from google.colab import files
files.download(str(ROOT/'Healthcare_KG_Population_Intelligence_V4_1_Outputs.zip'))

### Next decision gate
Only after reviewing the R comparisons, adjusted jackknife stability, CDC weight guidance for 2021–2023, and laboratory method documentation should we move to V5. The **overall gate deliberately remains REVIEW_REQUIRED** until the epidemiological review is completed.